# Etapa 1:

## Qual o problema socioeconômico que você está investigando?

O Nordeste apresenta um clima bem heterogêneo junto de uma biodiversidade ambiental vasta, tendo uma diferença grande nos regimes de precipitação, temperatura, umidade e disponibilidade hídrica. Essa variação pode afetar justamente as queimadas através de ressecamento da vegetação e do material de combustível, provocando maior intensidade nos focos de queimadas. Nesse contexto, o problema socioeconômico investigado está relacionado aos impactos que esses eventos podem provocar sobre a população e sobre as atividades econômicas da região, especialmente na agricultura, pecuária, saúde pública e conservação dos recursos naturais. Queimadas de maior intensidade podem causar perdas de áreas produtivas, degradação ambiental, aumento da emissão de poluentes atmosféricos e maior demanda por ações de combate e prevenção por parte do poder público.

## Por que ele é relevante para o Nordeste brasileiro?

Por causa da presença de extensas áreas sujeitas a períodos de estiagem e elevada variabilidade das condições meteorológicas, além da importância das atividades agropecuárias para diversos municípios da região. A combinação entre baixa precipitação, temperaturas elevadas, baixa umidade e disponibilidade de material combustível pode favorecer condições de maior risco de fogo. Dessa forma, a análise conjunta dos dados meteorológicos disponibilizados pelo Instituto Nacional de Meteorologia (INMET) e dos dados de focos de calor e Potência Radiativa do Fogo (FRP) do Instituto Nacional de Pesquisas Espaciais (INPE), entre 2020 e 2024, permite identificar padrões espaciais e temporais, períodos críticos e áreas com maior frequência ou intensidade de queimadas.

## Que tipo de decisão um gestor público poderia tomar com base nos resultados do seu modelo?

A identificação de períodos e regiões com maior probabilidade de ocorrência ou intensidade de queimadas poderia orientar a distribuição de equipes de combate a incêndios, a intensificação da fiscalização, a definição de áreas prioritárias para monitoramento e a emissão de alertas preventivos. Além disso, os resultados poderiam auxiliar no planejamento de políticas ambientais, agrícolas e de proteção civil, permitindo que os recursos públicos sejam direcionados de forma mais eficiente para os locais e períodos de maior risco.

## Hipótese inicial:

A precipitação acumulada nos dias anteriores (7, 15 ou 30 dias) à ocorrência do foco apresenta associação negativa mais forte com o FRP do que a precipitação registrada no próprio dia, de modo que períodos antecedentes mais secos estão associados a focos de maior intensidade.

A precipitação acumulada nos 7, 15 ou 30 dias anteriores à ocorrência do foco apresentará associação negativa estatisticamente significativa com o FRP (p < 0,05), sendo sua correlação, em valor absoluto, superior à observada entre a precipitação do próprio dia e o FRP.

A hipótese será rejeitada caso nenhuma das janelas de precipitação antecedente apresente associação negativa significativa com o FRP ou caso sua associação não seja superior à da precipitação registrada no próprio dia.

## Hipóteses secundárias:

### Hipótese 1:

Áreas caracterizadas por condições recorrentes de baixa precipitação e baixa umidade apresentam agrupamentos espaciais de focos com FRP elevado, indicando que a intensidade das queimadas não se distribui aleatoriamente no território nordestino.

Áreas classificadas no quartil inferior de precipitação acumulada e umidade relativa apresentarão maior proporção de focos classificados no quartil superior de FRP e autocorrelação espacial positiva e estatisticamente significativa, medida pelo índice de Moran global (I > 0; p < 0,05) e Moran Local/LISA.

A hipótese será rejeitada caso não seja detectada autocorrelação espacial positiva significativa ou caso as áreas de menor precipitação e umidade não apresentem maior concentração de focos com FRP elevado.

### Hipótese 2:

Velocidades mais elevadas do vento estão associadas a maiores valores de FRP, e essa associação é intensificada quando ocorrem simultaneamente baixa umidade relativa e baixa precipitação acumulada nos dias anteriores.

A velocidade do vento apresentará associação positiva e estatisticamente significativa com o FRP (p < 0,05), e o efeito estimado do vento será maior nas observações simultaneamente classificadas no quartil inferior de umidade relativa e precipitação antecedente do que nas demais condições.

A hipótese será rejeitada caso a velocidade do vento não apresente associação positiva significativa com o FRP ou caso o termo de interação entre vento e condições secas não indique aumento significativo do FRP.

# Etapa 2

In [12]:
# criando seção spark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('INPE-MLlib')
    .master('local[*]')
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    .getOrCreate()
)

Vamos ler primeiramente os dados de ambos as fontes (inpe e inmet) e analisar os primeiros registros

In [30]:
df_inpe = (
    spark.read
    .option("header", True)
    .option('sep', ',')
    .option('inferSchema', True)
    .csv("../data/bronze/inpe/*.csv")
)

In [33]:
df_inpe.show()

+-------------------+---------+------+--------------+------------------+--------------+-----------+------------+---------+----+-------------------+-------------------+
|           DataHora| Satelite|  Pais|        Estado|         Municipio|         Bioma|DiaSemChuva|Precipitacao|RiscoFogo| FRP|           Latitude|          Longitude|
+-------------------+---------+------+--------------+------------------+--------------+-----------+------------+---------+----+-------------------+-------------------+
|2020-01-01 00:45:28|  METOP-B|Brasil|        PARANÁ|         ITAPERUÇU|Mata Atlântica|       11.0|         3.3|      0.2|NULL|-25.187599182128906| -49.39350128173828|
|2020-01-01 00:45:28|  METOP-B|Brasil|        PARANÁ| RIO BRANCO DO SUL|Mata Atlântica|       11.0|         3.2|      0.2|NULL|-25.185699462890625|-49.383201599121094|
|2020-01-01 00:45:53|  METOP-B|Brasil|        PARANÁ|         ITAPERUÇU|Mata Atlântica|       11.0|         3.4|      0.2|NULL|-25.225500106811523| -49.38520050

In [36]:
df_inpe.printSchema()

root
 |-- DataHora: timestamp (nullable = true)
 |-- Satelite: string (nullable = true)
 |-- Pais: string (nullable = true)
 |-- Estado: string (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- Bioma: string (nullable = true)
 |-- DiaSemChuva: double (nullable = true)
 |-- Precipitacao: double (nullable = true)
 |-- RiscoFogo: double (nullable = true)
 |-- FRP: double (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)



Percebe-se que a coluna de DataHora veio como string, vamos tratar isso e transformar em timestamp:

DataHora: timestamp
DiaSemChuva: integer
Precipitacao: double
RiscoFogo: double
FRP (Fire Radiative Power): double
Latitude: double
Longitude: double

In [32]:
from pyspark.sql import functions as F

df_inpe = df_inpe.withColumn(
    "DataHora",
    F.coalesce(
        F.try_to_timestamp(
            F.regexp_replace(F.col("DataHora"), r"^[^\d]+", ""),
            F.lit("yyyy/MM/dd HH:mm:ss")
        ),
        F.try_to_timestamp(
            F.regexp_replace(F.col("DataHora"), r"^[^\d]+", ""),
            F.lit("yyyy-MM-dd HH:mm:ss")
        )
    )
)

In [11]:
df_inmet = (
    spark.read
    .option("header", True)
    .option('sep', ';')
    .option('inferSchema', True)
    .csv("../data/bronze/inmet/*.csv")
)

baixar os dados:
inpe -> baixe de https://data.inpe.br/queimadas/bdqueimadas/#exportar-dados, selecione apenas estados do nordeste e periodo 2020 a 2024, baixe um zip por ano, depois extraia os csvs e coloque na pasta data/bronze/

inmet -> baixe de https://bdmep.inmet.gov.br/, siga o passo a passo, selecione virgula, dados horários, estações automáticas, selecione região